In [95]:
import warnings

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

##### Decision Trees and Random Forests - Machine Learning with Python and scikit-learn #####
# The following topics are covered in this tutorial:
#    - Downloading a real-world dataset
#    - Preparing a dataset for training
#    - Training and interpreting decision trees
#    - Training and interpreting random forests
#    - Overfitting, hyperparameter tuning & regularization
#    - Making predictions on single inputs

### Problem Statement
# This tutorial takes a practical and coding-focused approach. We'll learn how to use decision trees and random forests to solve a real-world problem from Kaggle:
# QUESTION: The Rain in Australia dataset contains about 10 years of daily weather observations from numerous Australian weather stations.
# As a data scientist at the Bureau of Meteorology, you are tasked with creating a fully-automated system that can use today's weather data
# for a given location to predict whether it will rain at the location tomorrow.

# Let's install and import some required libraries before we begin.
#!pip install opendatasets legacy-cgi --quiet

import opendatasets as od
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib
import os
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
sns.set_style('darkgrid')
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

### Downloading the Data
# The dataset is available at https://www.kaggle.com/jsphyg/weather-dataset-rattle-package .
# We'll use the opendatasets library to download the data from Kaggle directly.

#od.download('https://www.kaggle.com/jsphyg/weather-dataset-rattle-package')
os.listdir('weather-dataset-rattle-package')
raw_df = pd.read_csv('weather-dataset-rattle-package/weatherAUS.csv')
#raw_df

# Each row shows the measurements for a given date at a given location. The last column "RainTomorrow" contains the value to be predicted.
# Let's check the column types of the dataset.
#raw_df.info()

# Let's drop any rows where the value of the target column RainTomorrow in empty.
raw_df.dropna(subset=['RainTomorrow'], inplace=True)


### EXERCISE: Perform exploratory data analysis on the dataset and study the relationship of other columns with the RainTomorrow column.


### Preparing the Data for Training
# We'll perform the following steps to prepare the dataset for training:
#  1. Create a train/test/validation split
#  2. Identify input and target columns
#  3. Identify numeric and categorical columns
#  4. Impute (fill) missing numeric values
#  5. Scale numeric values to the (0, 1) range
#  6. Encode categorical columns to one-hot vectors


### Training, Validation and Test Sets
#plt.title('No. of Rows per Year')
#sns.countplot(x=pd.to_datetime(raw_df.Date).dt.year);

# While working with chronological data, it's often a good idea to separate the training, validation and test sets with time, so that the model is trained
# on data from the past and evaluated on data from the future.
# We'll use the data till 2014 for the training set, data from 2015 for the validation set, and the data from 2016 & 2017 for the test set.

year = pd.to_datetime(raw_df.Date).dt.year

train_df = raw_df[year < 2015]
val_df = raw_df[year == 2015]
test_df = raw_df[year > 2015]
#print('train_df.shape :', train_df.shape)
#print('val_df.shape :', val_df.shape)
#print('test_df.shape :', test_df.shape)


### Input and Target Columns
# Let's identify the input and target columns.

input_cols = list(train_df.columns)[1:-1]
target_col = 'RainTomorrow'
train_inputs = train_df[input_cols].copy()
train_targets = train_df[target_col].copy()
val_inputs = val_df[input_cols].copy()
val_targets = val_df[target_col].copy()
test_inputs = test_df[input_cols].copy()
test_targets = test_df[target_col].copy()


### Let's also identify the numeric and categorical columns.
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes('object').columns.tolist()
#print(numeric_cols)
#print(categorical_cols)


### Imputing missing numeric values
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy = 'mean').fit(raw_df[numeric_cols])

train_inputs[numeric_cols] = imputer.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = imputer.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = imputer.transform(test_inputs[numeric_cols])

#val_inputs[numeric_cols].isna().sum()


### Scaling Numeric Features
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler().fit(raw_df[numeric_cols])

train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = scaler.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

#train_inputs
#train_inputs.describe().loc[['min', 'max']]


### Encoding Categorical Data
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(raw_df[categorical_cols])
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))

train_inputs[encoded_cols] = encoder.transform(train_inputs[categorical_cols])
val_inputs[encoded_cols] = encoder.transform(val_inputs[categorical_cols])
test_inputs[encoded_cols] = encoder.transform(test_inputs[categorical_cols])

#train_inputs

# As a final step, let's drop the textual categorical columns, so that we're left with just numeric data.
X_train = train_inputs[numeric_cols + encoded_cols]
X_val = val_inputs[numeric_cols + encoded_cols]
X_test = test_inputs[numeric_cols + encoded_cols]

#X_train


### Training and Visualizing Decision Trees
# A decision tree in general parlance represents a hierarchical series of binary decisions:
# A decision tree in machine learning works in exactly the same way, and except that we let the computer figure out the optimal structure & hierarchy
# of decisions, instead of coming up with criteria manually.

# 1. Training
# We can use DecisionTreeClassifier from sklearn.tree to train a decision tree.
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, train_targets)

# An optimal decision tree has now been created using the training data.

# 2. Evaluation
# Let's evaluate the decision tree using the accuracy score.
from sklearn.metrics import accuracy_score, confusion_matrix

train_preds = model.predict(X_train)
#train_preds
#pd.value_counts(train_preds)


# The decision tree also returns probabilities for each prediction.
train_probs = model.predict_proba(X_train)
#train_probs

# Seems like the decision tree is quite confident about its predictions.
# Let's check the accuracy of its predictions.
#accuracy_score(train_targets, train_preds)
# output: 0.9999797955307714

# The training set accuracy is close to 100%! But we can't rely solely on the training set accuracy, we must evaluate the model on the validation set too.
# We can make predictions and compute accuracy in one step using model.score
#model.score(X_val, val_targets)
# output: 0.7921188555510418

# Although the training accuracy is 100%, the accuracy on the validation set is just about 79%, which is only marginally better then always predicting "No".

# The "Dumb" Baseline
#val_targets.value_counts() / len(val_targets)
# No: 0.788, Yes: 0.211

# It appears that the model has learned the training examples perfect, and doesn't generalize well to previously unseen examples.
# This phenomenon is called "overfitting", and reducing overfitting is one of the most important parts of any machine learning project.

# 78.8% of the days in the validation set are "No Rain".
# So a dumb model that just guesses "No Rain" every single day gets 78.8%.
# Your tree got 79.2%.
# It's only 0.4% smarter than a dumb model.

## Why did this happen? (The Overfitting Trap)
# Imagine a student who memorizes the exact answers to 100 practice questions. If you give them those exact 100 questions again, they get 100%.
# But if you give them a new question with different numbers, they fail.
# That is exactly what the Decision Tree did.
# A Decision Tree has no limit to how many questions it can ask. It kept splitting the data until every single training row was in its own little box.
# It memorized the data perfectly. But when a new day of weather came in that didn't match its memorized boxes, it guessed wrong.

## Note: A single Decision Tree will overfit the training data. It gets 99.9% on train but only 79.2% on validation.
#  This means it memorized the training data but failed to learn general patterns. This is called Overfitting.

### Visualization
# We can visualize the decision tree learned from the training data.
from sklearn.tree import plot_tree, export_text

#plt.figure(figsize=(30,15))
#plot_tree(model, feature_names=X_train.columns, max_depth=2, filled=True);

# Can you see how the model classifies a given input as a series of decisions? The tree is truncated here, but following any path from the
# root node down to a leaf will result in "Yes" or "No". Do you see how a decision tree differs from a logistic regression model?

# 1. Reading the Tree
# Look at the very top line:
#   |--- Humidity3pm <= 0.72
# This is the Root (the first and most important question). The tree found that humidity at 3 PM is the best single feature
# to split the data.
#    If the humidity is less than 0.72, it goes down the left branch.
#    If the humidity is greater than 0.72, it goes down the right branch.
# As you follow the branches down, it asks more questions: "Is rainfall 0? Is sunshine low? Is the pressure high?"
# At the very bottom (the leaves), it makes a final decision: class: Yes or class: No.

# 2. The gini Value (The Loss Function)
# Look at a box that says class: Yes or class: No. You'll see a gini value.
# Gini measures impurity (how mixed the classes are).
#    If a box has 50% Yes and 50% No, the Gini is 0.5 (very mixed).
#    If a box is 100% Yes and 0% No, the Gini is 0.0 (perfectly pure).
# The tree's entire goal is to ask questions that drop the Gini score to 0 as fast as possible.

## Conceptually speaking, while training the models evaluates all possible splits across all possible columns and picks the best one.
# Then, it recursively performs an optimal split for the two portions. In practice, however, it's very inefficient to check all possible
# splits, so the model uses a heuristic (predefined strategy) combined with some randomization.

# Let's check the depth of the tree that was created.
#model.tree_.max_depth
# ouput: 48

# The tree is 48 layers deep.
# To put that in perspective, a normal, healthy tree is usually 3 to 10 layers deep.
# Why did it go 48 layers deep? Because it's overfitting. It kept asking questions until it memorized every single row in the training set.
# It asked 48 questions just to figure out one day's weather!

# We can also display the tree as text, which can be easier to follow for deeper trees.
#tree_text = export_text(model, max_depth=10, feature_names=list(X_train.columns))
#print(tree_text[:5000])

## Note: The Decision Tree visualizes its logic as a flowchart of Yes/No questions. It uses the Gini Index to find the best split.
#  A Gini of 0 means a perfect split. Our tree went 48 layers deep, which is why it memorized the training data (overfitting).


### Feature Importance
# Based on the gini index computations, a decision tree assigns an "importance" value to each feature. These values can be used to
# interpret the results given by a decision tree.
#model.feature_importances_

## In Linear Regression, you looked at the coefficients (weights) to see which feature was most important.
# In a Decision Tree, you look at feature_importances_. It's a score between 0 and 1 that represents how much a feature helped
# reduce the Gini impurity (how many good splits it made).

# Let's turn this into a dataframe and visualize the most important features.
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
importance_df.head(10)

#plt.title('Feature Importance')
#sns.barplot(data=importance_df.head(10), x='importance', y='feature');

## Note: feature_importances_ shows which columns the tree used most to make splits. A score of 0.26 for Humidity3pm means it was by far
#  the most important feature. This is a huge advantage of Decision Trees: they are highly interpretable.


### Hyperparameter Tuning and Overfitting
# As we saw in the previous section, our decision tree classifier memorized all training examples, leading to a 100% training accuracy,
# while the validation accuracy was only marginally better than a dumb baseline model. This phenomenon is called overfitting, and in this section,
# we'll look at some strategies for reducing overfitting.

# The DecisionTreeClassifier accepts several arguments, some of which can be modified to reduce overfitting.
#DecisionTreeClassifier

# These arguments are called hyperparameters because they must be configured manually (as opposed to the parameters within the model which are
# learned from the data. We'll explore a couple of hyperparameters:
#   -> max_depth
#   -> max_leaf_nodes

## max_depth
# By reducing the maximum depth of the decision tree, we can prevent the tree from memorizing all training examples, which may lead to better generalization
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, train_targets)

# What it does: It forces the tree to stop growing after 3 layers of questions.
#    . By limiting the depth, you force the tree to focus only on the most important patterns (like Humidity and Pressure).
#    . It can't memorize the random noise anymore.
#    . The result: Training accuracy goes down (e.g., from 99% to 85%), but Validation accuracy goes up (e.g., from 79% to 84%).

# We can compute the accuracy of the model on the training and validation sets using model.score
model.score(X_train, train_targets)
# output: 0.8291308037337859

model.score(X_val, val_targets)
# output: 0.8334397307178921

## Great, while the training accuracy of the model has gone down, the validation accuracy of the model has increased significantly.

#model.classes_
#plt.figure(figsize=(80,20))
#plot_tree(model, feature_names=X_train.columns, filled=True, rounded=True, class_names=model.classes_);

#print(export_text(model, feature_names=list(X_train.columns)))


## Let's experiment with different depths using a helper function.
def max_depth_error(md):
    model = DecisionTreeClassifier(max_depth=md, random_state=42)
    model.fit(X_train, train_targets)
    train_acc = 1 - model.score(X_train, train_targets)
    val_acc = 1 - model.score(X_val, val_targets)
    return {'Max Depth': md, 'Training Error': train_acc, 'Validation Error': val_acc}

# This function takes a depth (md). It trains a new tree with that depth and calculates the error.
# Note: 1 - model.score() just means "Error". If accuracy is 85%, error is 15%.

errors_df = pd.DataFrame([max_depth_error(md) for md in range(1, 21)])

# This is a Python list comprehension. It runs the function 20 times, testing max_depth from 1 to 20. It then puts the results into a nice table.
#errors_df

#plt.figure()
#plt.plot(errors_df['Max Depth'], errors_df['Training Error'])
#plt.plot(errors_df['Max Depth'], errors_df['Validation Error'])
#plt.title('Training vs. Validation Error')
#plt.xticks(range(0,21, 2))
#plt.xlabel('Max. Depth')
#plt.ylabel('Prediction Error (1 - Accuracy)')
#plt.legend(['Training', 'Validation'])

# This is a common pattern you'll see with all machine learning algorithms:
# This draws two lines on a graph:
#  - Training Error: Will keep going down (towards 0) as depth increases.
#  - Validation Error: Will go down, hit a bottom, and then start going back up (overfitting).

## This "U-shaped curve" is the most famous concept in ML. It tells you when a model has learned enough and when it has learned too much.
# You'll often need to tune hyperparameters carefully to find the optimal fit. In the above case, it appears that a maximum depth of 7
# results in the lowest validation error.

## The Sweet Spot
model = DecisionTreeClassifier(max_depth=7, random_state=42).fit(X_train, train_targets)
#model.score(X_val, val_targets)
# output: 0.8453949277465034

# The graph shows that at depth 7, the validation error is at its absolute lowest. So, the author trains the final model with max_depth=7 and
# checks its score on the validation set.


### max_leaf_nodes
# Another way to control the size of complexity of a decision tree is to limit the number of leaf nodes.
# This allows branches of the tree to have varying depths.
# If max_depth is like saying "No branch can be longer than 7 steps", max_leaf_nodes is like saying "The whole tree can only have 128 final endings (leaves)."

## Why is this better sometimes?
# Imagine a tree where the left side is very simple (it stops after 2 questions), but the right side is very complex (it goes 12 questions deep).
# max_depth would force the right side to stop at 7, even if it needs more.
# max_leaf_nodes allows the right side to go 12 levels deep, as long as it doesn't exceed the total limit of 128 leaves.
model = DecisionTreeClassifier(max_leaf_nodes=128, random_state=42)
model.fit(X_train, train_targets)

#model.score(X_train, train_targets)
# output: 0.8480421869317493

#model.score(X_val, val_targets)
# output: 0.8442342290058615

model.tree_.max_depth

# Look at the output of model.tree_.max_depth. It should be around 12. This proves that the tree grew deeper than 7 in some places,
# but still kept the overall size in check to prevent overfitting.


## -> Hyperparameters are settings we choose before training (like max_depth, n_estimators, random_state).
## -> Parameters are the numbers the model learns during training (like weights in regression, or the exact splits in a tree).



##### Training a Random Forest #####
# While tuning the hyperparameters of a single decision tree may lead to some improvements, a much more effective strategy is to combine the results of several
# decision trees trained with slightly different parameters. This is called a random forest.

# The key idea here is that each decision tree in the forest will make different kinds of errors, and upon averaging, many of their errors will cancel out.
# This idea is also known as the "wisdom of the crowd" in common parlance.
## A random forest works by averaging/combining the results of several decision trees

# We'll use the RandomForestClassifier class from sklearn.ensemble.
from sklearn.ensemble import RandomForestClassifier

#model = RandomForestClassifier(n_estimators=100, random_state=42)
# A Random Forest isn't just one tree. It's a whole forest of them. When you write RandomForestClassifier(), Scikit-learn silently builds
# 100 different Decision Trees.
# When you ask the model to predict "Will it rain?", all 100 trees vote. If 65 trees say "Yes" and 35 say "No", the forest predicts "Yes".

model = RandomForestClassifier(n_jobs=1, random_state=42)
# n_jobs=-1: This tells the computer to use all of your CPU cores to train the trees in parallel. It makes it incredibly fast. (same default 100)

model.fit(X_train, train_targets)

#model.score(X_train, train_targets)
# output: 0.9999494888269285

model.score(X_val, val_targets)
# output: 0.8566537055307295

# You will see the Training score is 100% (just like the single tree), but the Validation score will jump to ~85% or 86%. It beats the single tree!
# This is the "Wisdom of the Crowd."

# This general technique of combining the results of many models is called "ensembling",
# it works because most errors of individual models cancel out on averaging.

# We can also look at the probabilities for the predictions. The probability of a class is simply the fraction of trees which that predicted the given class.
train_probs = model.predict_proba(X_train)
#train_probs

# Because there are 100 trees, the probability is now calculated by voting. If 80 trees say "Yes" and 20 trees say "No", the probability is 80%.
# (With a single tree, it was always exactly 0 or 1).

## We can can access individual decision trees using model.estimators_
#len(model.estimators_)

# This will print 100. The model contains an array of 100 separate decision trees. You can even plot them individually:
# You can even plot them individually:

#plt.figure(figsize=(80,20))
#plot_tree(model.estimators_[15], max_depth=2, feature_names=X_train.columns, filled=True, rounded=True, class_names=model.classes_);

## Note: Each tree in a Random Forest is trained on a random subset of rows and features. In Tree 15, the root node uses Sunshine, and it only sees 62,448 samples (not the full training set).
#  This randomness ensures the trees make different errors.

## Note: A Random Forest trains 100 decision trees and averages their predictions. Because each tree makes different errors, the errors cancel out when
#  averaged (Wisdom of the Crowd). n_jobs=-1 uses all CPU cores to train them in parallel.

## Just like decision tree, random forests also assign an "importance" to each feature, by combining the importance values from individual trees.
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# It grabs the feature importance from the Random Forest. Because the forest has 100 trees, the importance is just the average of all 100 trees.
# It then sorts them and plots the top 10.

#importance_df.head(10)

#plt.title('Feature Importance')
#sns.barplot(data=importance_df.head(10), x='importance', y='feature');

# Notice that the distribution is a lot less skewed than that for a single decision tree.
# Remember the single Decision Tree? Humidity3pm had an importance of 0.26 (26%), and all the other features were tiny (1% or 2%).
# It was heavily skewed toward one feature.
## In the Random Forest, the top feature might only be 0.15 (15%), and the next 5 features might be 0.10 (10%) each.
# The importance is spread out more evenly among the top features.

## Note: Random Forest feature importance is the average of all 100 trees. Because each tree sees a random subset of features,
#  the model relies on a wider variety of features. This makes the importance distribution less skewed (more balanced) than a single Decision Tree.


##### Hyperparameter Tuning with Random Forests
# Just like decision trees, random forests also have several hyperparameters. In fact many of these hyperparameters are applied to the underlying decision trees.
# Let's create a base model with which we can compare models with tuned hyperparameters.
base_model = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_train, train_targets)

# This trains the default forest: 100 trees. It saves the training and validation accuracy as your "baseline" to beat.
base_train_acc = base_model.score(X_train, train_targets)
base_val_acc = base_model.score(X_val, val_targets)

base_accs = base_train_acc, base_val_acc
#base_accs
# output: (0.9999494888269285, 0.8566537055307295)

## n_estimators
# This controls the number of decision trees in the random forest. The default value is 100. For larger datasets, it helps to have a greater number
# of estimators. As a general rule, try to have as few estimators as needed.
#model = RandomForestClassifier(random_state=42, n_jobs=-1, n_estimators=10)
#model.fit(X_train, train_targets)
#model.score(X_train, train_targets), model.score(X_val, val_targets)
# output: (0.986958015112943, 0.8485868492832686)

# This builds a forest with only 10 trees.
# What happens: The validation accuracy will likely drop slightly. With only 10 trees, the "wisdom of the crowd" isn't as strong,
# and the random errors don't cancel out as well. It's also much faster to train.

#model = RandomForestClassifier(random_state=42, n_jobs=-1, n_estimators=500)
#model.fit(X_train, train_targets)
#model.score(X_train, train_targets), model.score(X_val, val_targets)
# output: (0.9999797955307714, 0.8577563693343393)

# This builds a forest with 500 trees.
# What happens: The validation accuracy might go up by a tiny amount (maybe 0.1% or 0.2%). But it takes 5 times longer to train.

## Note: n_estimators controls the number of trees in the forest. More trees improve accuracy slightly, but with diminishing returns.
# 10 trees are too few; 500 trees are slow. The default of 100 is usually the sweet spot.


## max_features
# Instead of picking all features for every split, we can specify only a fraction of features to be chosen randomly.
# The Problem: Why not use all features?
# Imagine you have a dataset where Humidity is the absolute best feature for predicting rain.
# If every single tree in your forest is allowed to look at every single feature, then every single tree will choose Humidity for its first split.
# The trees will all look identical. It’s no longer a "forest"; it's just 100 copies of the same tree. If that tree is wrong, they are all wrong.

# The Solution: max_features
# max_features forces each tree to hide most of the features from itself at every split.
#test_params(max_features='log2')

# It's a mathematical shortcut. If you have 119 features, log2(119) is roughly 6.8. So telling Scikit-learn max_features='log2' means
# "Let each tree look at about 6 or 7 random features at a time."

#test_params(max_features=3)
#test_params(max_features=6)

# max_features=3: Every time a tree wants to split, it can only look at 3 random features (out of your 119). It might get lucky and see Humidity.
# Or it might only see WindSpeed, Temp, and Cloud.
# If it doesn't see Humidity, it is forced to find a different way to split the data.

#Let's define a helper function to max_features and other hyperparameters easily.
def test_params(**params):
    model = RandomForestClassifier(random_state=42, n_jobs=-1, **params).fit(X_train, train_targets)
    train_acc = model.score(X_train, train_targets)
    val_acc = model.score(X_val, val_targets)
    print(f"Params: {params}")
    print(f"Train Accuracy: {train_acc:.4f}, Val Accuracy: {val_acc:.4f}\n")

#test_params(max_features='log2')
# output: Train Accuracy: 1.0000, Val Accuracy: 0.8559

#test_params(max_features=3)
# output: Train Accuracy: 0.9999, Val Accuracy: 0.8543

#test_params(max_features=6)
# output: Train Accuracy: 1.0000, Val Accuracy: 0.8559

# Why is this good?
# Because it forces the trees to be diverse.
# Tree 1 might learn to predict rain based on Humidity.
# Tree 2 might learn to predict rain based on Pressure.
# Tree 3 might learn to predict rain based on Sunshine.
# When they all vote, they are bringing completely different perspectives to the table. That is true "Wisdom of the Crowd."

## Note: max_features limits how many features a tree can look at during a split. By hiding features randomly, it forces the trees to be different
#  from each other. If all trees saw all features, they would all be identical. Diversity makes the forest strong.


## max_depth and max_leaf_nodes
# These arguments are passed directly to each decision tree, and control the maximum depth and max. no leaf nodes of each tree respectively.
# By default, no maximum depth is specified, which is why each tree has a training accuracy of 100%. You can specify a max_depth to reduce overfitting.

#test_params(max_depth=5)
# output: Train Accuracy: 0.8198, Val Accuracy: 0.8241

# max_depth=5: This forces every tree in the forest to be very short. It will stop overfitting completely, but it might be too simple (Underfitting).
# Both train and val accuracy will be lower.

#test_params(max_depth=25)
# output: Train Accuracy: 0.9776, Val Accuracy: 0.8559

# max_depth=25: This lets the trees grow quite deep. It might find more patterns, but it might also start memorizing.

#test_params(max_leaf_nodes=2**5)
# output: Train Accuracy: 0.8314, Val Accuracy: 0.8339

# 2**5 (32 leaves): Very simple tree. Similar to max_depth=5.

#test_params(max_leaf_nodes=2**20)
# output: Train Accuracy: 1.0000, Val Accuracy: 0.8568

# 2**20 (1,048,576 leaves): Basically unlimited leaves. The trees will grow until they memorize the data (100% training accuracy).

# The tutorial says: "The optimal values of max_depth and max_leaf_nodes lies somewhere between 0 and unbounded."
# This means you have to play Goldilocks:
#   -> If you limit the trees too much, they underfit.
#   -> If you don't limit them at all, they overfit (even in a forest).
#   -> The sweet spot is somewhere in the middle.

## Note: We can pass max_depth and max_leaf_nodes to a Random Forest. This applies the limit to every single tree in the forest.
#  Finding the right limit prevents the individual trees from memorizing the data, which improves the overall validation accuracy.


## min_samples_split and min_samples_leaf
# By default, the decision tree classifier tries to split every node that has 2 or more. You can increase the values of these arguments
# to change this behavior and reduce overfitting, especially for very large datasets.

#test_params(min_samples_split=3, min_samples_leaf=2)
# output: Train Accuracy: 0.9625, Val Accuracy: 0.8566
# This is very close to default. The tree will grow deep.

#test_params(min_samples_split=100, min_samples_leaf=60)
# output: Train Accuracy: 0.8496, Val Accuracy: 0.8451
# This is a heavy restriction. The tree won't be able to grow very deep. Training accuracy will drop (maybe 85%),
# but Validation accuracy might stay the same or even go up slightly because it's not allowed to overfit.

# Let's break down what they mean:
#  1. min_samples_split (The "Don't bother" Rule)
#   By default, if a box has 2 or more samples, the tree will try to split it.
#     - If you set min_samples_split=100, the tree is not allowed to split a box unless it has at least 100 samples in it.
#     -  Why? If a box only has 5 people in it, splitting it is just memorizing those 5 specific people. It's better to leave them together in one box.
#  2. min_samples_leaf (The "Minimum Box Size" Rule)
#   This controls the size of the final boxes at the bottom of the tree.
#     - If you set min_samples_leaf=60, the tree is not allowed to create a final box (leaf) unless that box contains at least 60 samples.
#     - Why? It stops the tree from creating tiny rules like "If humidity is 82.1%, temperature is 23.4, and the wind is North, it rains."
#       (which only applies to 1 day). By requiring at least 60 samples, the rule must apply to a decent chunk of the population.

## Note: min_samples_split and min_samples_leaf prevent the tree from creating tiny rules based on just a few samples.
# Increasing them reduces overfitting and makes the model more generalized.


## min_impurity_decrease
# This argument is used to control the threshold for splitting nodes. A node will be split if this split induces a decrease of the impurity (Gini index)
#  greater than or equal to this value. It's default value is 0, and you can increase it to reduce overfitting.

#test_params(min_impurity_decrease=1e-6)
# output: Train Accuracy: 0.9888, Val Accuracy: 0.8561

#test_params(min_impurity_decrease=1e-2)
# output: Train Accuracy: 0.7749, Val Accuracy: 0.7883

# What it is: The "Why Bother?" rule.
# How it works: If splitting a box only makes the data 0.001% purer, the tree usually does it anyway.
# By setting min_impurity_decrease=1e-2, you tell the tree: "Don't bother splitting unless it makes the data at least 1% purer."
# Result: It stops the tree from making tiny, useless splits. It reduces overfitting.

## bootstrap, max_samples
# By default, a random forest doesn't use the entire dataset for training each decision tree. Instead it applies a technique called bootstrapping.
# For each tree, rows from the dataset are picked one by one randomly, with replacement
# i.e. some rows may not show up at all, while some rows may show up multiple times.

# Bootstrapping helps the random forest generalize better, because each decision tree only sees a fraction of th training set,
# and some rows randomly get higher weightage than others.

#test_params(bootstrap=False)
# output: Train Accuracy: 1.0000, Val Accuracy: 0.8568

# bootstrap=False: By default, every tree in the forest sees a random 63% of the data (with replacement).
# If you set bootstrap=False, every tree sees the entire dataset. They will all look identical, and the "Wisdom of the Crowd" magic disappears.
# (Don't do this unless you have a very specific reason).

#test_params(max_samples=0.9)
# output: Train Accuracy: 0.9998, Val Accuracy: 0.8565

# max_samples=0.9: If bootstrap is True, you can control exactly how much data each tree gets. 0.9 means each tree gets a random 90% of the rows.


## class_weight
# The Problem: Remember when we saw that "No Rain" is much more common than "Yes Rain"? The model might just get lazy and always predict
# "No" because it's usually right.
# class_weight='balanced': This tells the model: "Hey, 'Yes' and 'No' are equally important. If you miss a 'Yes', it's a huge penalty."
# It automatically adjusts the weights so the model pays more attention to the minority class.
# class_weight={'No': 1, 'Yes': 2}: You are manually telling the model that missing a "Yes" is twice as bad as missing a "No".

#test_params(class_weight='balanced')
# output:

#test_params(class_weight={'No': 1, 'Yes': 2})
# output:


## Putting it together
# This is the grand finale! You are taking everything you learned about tuning and building the ultimate Random Forest.
model = RandomForestClassifier(
    n_jobs=-1,                   # Use all CPU cores (fast)
    random_state=42,             # Reproducibility
    n_estimators=200,            # 200 trees (more than the default 100)
    max_features=7,              # Each split looks at 7 random features
    max_depth=30,                # Limit the depth to 30 to prevent overfitting
    class_weight={'No': 1, 'Yes': 2} # Penalize missing a "Yes" (rain) more than a "No"
)

model.fit(X_train, train_targets)
#model.score(X_train, train_targets), model.score(X_val, val_targets)
#base_accs = (0.9999494888269285, 0.8566537055307295)

# Note: We combined all the hyperparameter tuning into one final model. By increasing n_estimators, limiting max_depth, and adjusting class_weight,
# we improved the validation accuracy to ~85.7%. This is near the physical limit for this dataset because weather is inherently chaotic.

### 📌 Note: The Reality of Hyperparameter Tuning
# My Results:
#    - Base Model: Train 99.99% | Val: 85.66%
#    - Final Tuned Model: Train 99.41% | Val: 85.60%

# Why did the tuned model perform slightly worse (or exactly the same)?
#  1. The "Noise Ceiling": Weather is chaotic. There is a physical limit to how accurately we can predict it with this data. The model has already
#     extracted all the useful patterns.
#  2. Over-Tuning: By adding strict rules (like class_weight and max_depth), we restricted the trees. While the train accuracy dropped (good),
#     it didn't magically find new patterns in the validation set.
#  3. Diminishing Returns: Going from a basic model to a complex one might only yield a 0.1% difference, or in some cases, a negative difference.

# The Golden Takeaway:
# More tuning does not always equal a better model. Always compare your complex, heavily-tuned models against a simple baseline.
# Often, the default settings are already in the "sweet spot."

"""
Random Forest Hyperparameter Cheat Sheet

**1. Tree Size & Complexity (Controls Overfitting)**
- **`max_depth`**: Limits how many questions a tree can ask. (e.g., `max_depth=30`). Prevents memorizing data.
- **`max_leaf_nodes`**: Limits the total number of final boxes/leaves. (e.g., `max_leaf_nodes=128`).
- **`min_samples_split`**: The minimum number of samples a box must have before it's allowed to split. (e.g., `min_samples_split=3`).
- **`min_samples_leaf`**: The minimum number of samples required to be in a final leaf box. (e.g., `min_samples_leaf=2`).
- **`min_impurity_decrease`**: A node will only split if it decreases the Gini impurity by at least this amount. Stops useless splits.

**2. Forest Diversity (The "Wisdom of the Crowd")**
- **`n_estimators`**: The number of trees in the forest. Default is 100. (e.g., `n_estimators=500`). More trees = slight accuracy boost, but slower training.
- **`max_features`**: How many random columns a tree can look at per split. (e.g., `max_features=7` or `'log2'`). Forces trees to be different from each other.
- **`bootstrap`**: `True` by default. Each tree trains on a random subset of the data (with replacement).
- **`max_samples`**: If `bootstrap=True`, this controls the size of that subset (e.g., `max_samples=0.9` means 90% of the data).

**3. Handling Imbalance**
- **`class_weight`**: Adjusts the penalty for getting a class wrong. (e.g., `class_weight={'No': 1, 'Yes': 1.5}` means missing a "Yes" is 1.5x worse).

**4. Speed**
- **`n_jobs`**: `-1` uses all CPU cores to train trees in parallel.

**The Reality of Tuning:**
While these settings can reduce overfitting, they don't always improve validation accuracy. If the data is noisy (like weather), tuning might just lower
training accuracy without improving validation accuracy.
"""


### Making Predictions on New Inputs
def predict_input(single_input):
    input_df = pd.DataFrame([single_input])
    input_df[numeric_cols] = imputer.transform(input_df[numeric_cols])
    input_df[numeric_cols] = scaler.transform(input_df[numeric_cols])
    input_df[encoded_cols] = encoder.transform(input_df[categorical_cols])
    X_input = input_df[numeric_cols + encoded_cols]
    pred = model.predict(X_input)[0]
    prob = model.predict_proba(X_input)[0][list(model.classes_).index(pred)]
    return pred, prob

new_input = {
    'Date': '2021-06-19',
    'Location': 'Launceston',
    'MinTemp': 23.2,
    'MaxTemp': 33.2,
    'Rainfall': 10.2,
    'Evaporation': 4.2,
    'Sunshine': np.nan,
    'WindGustDir': 'NNW',
    'WindGustSpeed': 52.0,
    'WindDir9am': 'NW',
    'WindDir3pm': 'NNE',
    'WindSpeed9am': 13.0,
    'WindSpeed3pm': 20.0,
    'Humidity9am': 89.0,
    'Humidity3pm': 58.0,
    'Pressure9am': 1004.8,
    'Pressure3pm': 1001.5,
    'Cloud9am': 8.0,
    'Cloud3pm': 5.0,
    'Temp9am': 25.7,
    'Temp3pm': 33.0,
    'RainToday': 'Yes'
}

predict_input(new_input)

('Yes', np.float64(0.7409350005119366))